In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

clean_customer = spark.table("default.clean_customer")
raw_orders = spark.table("default.raw_orders")
raw_products = spark.table("default.raw_products")

In [0]:
clean_customer.printSchema()

raw_orders.printSchema()

raw_products.printSchema()

root
 |-- Customer_ID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- OrderID: string (nullable = true)
 |-- CustID: string (nullable = true)
 |-- Amount: double (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)

root
 |-- ProductID: string (nullable = true)
 |-- ProductName: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Price: long (nullable = true)
 |-- Load_Date: timestamp (nullable = true)
 |-- Record_Source: string (nullable = true)
 |-- Batch_ID: integer (nullable = true)



In [0]:
hub_customer = clean_customer.select(
    sha2(col("Customer_ID"),256).alias("Customer_HK"),
    col("Customer_ID"),
    col("Load_Date"),
    col("Record_Source")
).dropDuplicates(["Customer_ID"])

In [0]:
hub_customer.show(5,False)

+----------------------------------------------------------------+-----------+--------------------------+-------------+
|Customer_HK                                                     |Customer_ID|Load_Date                 |Record_Source|
+----------------------------------------------------------------+-----------+--------------------------+-------------+
|a4906a54b9b8416b9a8136e3665eb9d5c1cab6e72830f662d9b80a0d68b46b07|C001       |2026-07-13 10:01:07.445381|Customer_CSV |
|fdc98571fc93e8482954dcc8034bda27f571bd691cd86510a849dce99759933d|C002       |2026-07-13 10:01:07.445381|Customer_CSV |
|64604487c74ef2fb0aca58ed3e951aacc1f537baf96c023b75441012e4b92880|C003       |2026-07-13 10:01:07.445381|Customer_CSV |
|f12ce50422ff0b232d51ad5764fad492b2f6bc3f1a74f3e5c04cc5d2742f9605|C004       |2026-07-13 10:01:07.445381|Customer_CSV |
|95adbabc0d9bd895d140d98d75be9ad39f4d91cc7a53b37cfdfac32f5b75a332|C005       |2026-07-13 10:01:07.445381|Customer_CSV |
+---------------------------------------

In [0]:
%sql
DROP TABLE IF EXISTS default.hub_customer;

In [0]:
hub_customer.write.mode("overwrite").saveAsTable("default.hub_customer")

In [0]:
%sql
SELECT COUNT(*) FROM default.hub_customer;

COUNT(*)
100


In [0]:
hub_order = raw_orders.select(
    sha2(col("OrderID"),256).alias("Order_HK"),
    col("OrderID"),
    col("Load_Date"),
    col("Record_Source")
).dropDuplicates(["OrderID"])

In [0]:
hub_order.show(5,False)

+----------------------------------------------------------------+-------+-------------------------+-------------+
|Order_HK                                                        |OrderID|Load_Date                |Record_Source|
+----------------------------------------------------------------+-------+-------------------------+-------------+
|8d4de75f857fa7bb3c2b3129411332789d128f73d4321c12d696ea73f974f08b|O100   |2026-07-13 10:07:07.11227|Orders_CSV   |
|17dd225ead2a9bcf724a1e8acb9a3fc7e101c6696d33fe1e1013d62fb905aaeb|O101   |2026-07-13 10:07:07.11227|Orders_CSV   |
|d68b07bc90bfeccf42b22ae183c33e90c3ecac49de5ce797e4631ee8c1d5376d|O102   |2026-07-13 10:07:07.11227|Orders_CSV   |
|3ef092214a0da9ee2c9cb8981ee60bcaca4785691b485337dbc11db0c2f000ce|O103   |2026-07-13 10:07:07.11227|Orders_CSV   |
|3b9fe2267c3b1cbdcdde44005276bedc39210f5919678015664f8e9d8be421cb|O104   |2026-07-13 10:07:07.11227|Orders_CSV   |
+----------------------------------------------------------------+-------+------

In [0]:
%sql
DROP TABLE IF EXISTS default.hub_order;

In [0]:
hub_order.write.mode("overwrite").saveAsTable("default.hub_order")

In [0]:
%sql
SELECT COUNT(*) FROM default.hub_order;

COUNT(*)
500


In [0]:
hub_product = raw_products.select(
    sha2(col("ProductID"),256).alias("Product_HK"),
    col("ProductID"),
    col("Load_Date"),
    col("Record_Source")
).dropDuplicates(["ProductID"])

In [0]:
hub_product.show(5,False)

+----------------------------------------------------------------+---------+-------------------------+-------------+
|Product_HK                                                      |ProductID|Load_Date                |Record_Source|
+----------------------------------------------------------------+---------+-------------------------+-------------+
|df1e40051eff4bcf9b4ebc93083bfcad7f5195746b3e657de6b72bf3cb8897c3|P001     |2026-07-13 10:08:16.19356|Products_CSV |
|eea502719605f8ec96582d94a0f1738190fe72d16c7b57de48fcfcebfbcb631a|P002     |2026-07-13 10:08:16.19356|Products_CSV |
|12061cdd9a5c92ac2919a8567b74417f879ec93612941fb6bf4fdb941e7003eb|P003     |2026-07-13 10:08:16.19356|Products_CSV |
|00be3dd19f080c4255bba1ed6fcfbe7d8490f6a0299e336c906b212ee035bb80|P004     |2026-07-13 10:08:16.19356|Products_CSV |
|e370b726b942a9efa03d66cc60370da2bcd962fb69ceba7f8e3d8d3a27d2a42f|P005     |2026-07-13 10:08:16.19356|Products_CSV |
+---------------------------------------------------------------

In [0]:
%sql
DROP TABLE IF EXISTS default.hub_product;

In [0]:
hub_product.write.mode("overwrite").saveAsTable("default.hub_product")

In [0]:
%sql
SELECT COUNT(*) FROM default.hub_product;

COUNT(*)
20


In [0]:
%sql
SELECT COUNT(*) FROM default.hub_customer;

SELECT COUNT(*) FROM default.hub_order;

SELECT COUNT(*) FROM default.hub_product;

COUNT(*)
20
